# Análisis Exploratorio de Datos — Detección de Defectos en Placas PCB con IA

**Curso:** Análisis Exploratorio de Datos (Python · Pandas · Google Colab)

**Equipo:**

| Estudiante | Código | GitHub |
|---|---|---|
| Juan Diego Garzón Cordero | 2254270 | [@JD202730](https://github.com/JD202730) |
| Yusmary Tatiana Castañeda García | 2254254 | [@YusTati27](https://github.com/YusTati27) |
| Libardo Enrique Argumedo Mejía | 2264084 | — |

**Fecha:** Noviembre 2026

---

## Pregunta de investigación

> **¿Es viable usar modelos de IA (visión por computadora) para detectar automáticamente
> si una placa de circuito impreso (PCB) está bien o mal ensamblada/fabricada?**

## Dataset: DeepPCB

- **Fuente:** https://github.com/tangsanli5201/DeepPCB (Licencia MIT, uso académico)
- **Paper:** Ding, R. et al. *"TDD-net: a tiny defect detection network for printed circuit boards"*, CAAI Transactions on Intelligence Technology, 4(2), 2019.
- **Descripción:** 1,500 pares de imágenes de 640×640 px (~48 px/mm) tomadas con un escáner CCD lineal. Cada par contiene la imagen de una placa **sin defectos** (plantilla) y la misma placa **con defectos**, con anotaciones tipo bounding box.
- **6 tipos de defectos:**

| ID | Tipo | Descripción |
|---|---|---|
| 1 | `open` | Circuito abierto: traza interrumpida |
| 2 | `short` | Cortocircuito: cobre que conecta trazas que no debían unirse |
| 3 | `mousebite` | "Mordida de ratón": muesca en el borde de una traza |
| 4 | `spur` | Espolón: protuberancia puntiaguda de cobre |
| 5 | `spurious copper` | Cobre espurio: residuo de cobre donde no debe estar |
| 6 | `pin-hole` | Agujero de pin: perforación diminuta en una traza |

## Estructura del notebook

| Sección | Contenido |
|---|---|
| 1 | Configuración y descarga de datos |
| 2 | Carga de datos crudos (paso a paso) |
| 3 | **Limpieza y validación** (con evidencia antes/después) |
| 4 | **Análisis univariado** (estadísticas descriptivas) |
| 5 | **Análisis bivariado** (tamaño × tipo, posición × tipo) |
| 6 | **Visualizaciones** (4 gráficos + ejemplo visual) |
| 7 | Conclusiones, limitaciones y referencias

## 1. Configuración y descarga de datos

Clonamos el repositorio público del dataset directamente en Colab
(no requiere credenciales ni API de Kaggle).

In [ ]:
%%bash
if [ ! -d /content/DeepPCB ]; then
  git clone --depth 1 https://github.com/tangsanli5201/DeepPCB /content/DeepPCB
fi
du -sh /content/DeepPCB/PCBData

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

RUTA_DATOS = "/content/DeepPCB/PCBData"
print("Repositorio listo:", os.path.isdir(RUTA_DATOS))

MAPEO_TIPOS = {1: "open", 2: "short", 3: "mousebite", 4: "spur", 5: "spurious copper", 6: "pin-hole"}

## 2. Carga de datos crudos (paso a paso)

El dataset no viene en un CSV: son imágenes + archivos de anotación. El proceso de carga
es parte del trabajo:

1. Localizar los archivos `.txt` de anotación (uno por imagen con defectos).
2. Leer cada fila con formato `x1 y1 x2 y2 tipo`.
3. Construir un **DataFrame con una fila por defecto** y columnas derivadas (tamaño, centroide).
4. Guardar una copia cruda para comparar en la limpieza.

In [ ]:

def cargar_anotaciones(ruta_datos: str) -> pd.DataFrame:
    """Lee todos los archivos .txt de anotación y devuelve un DataFrame con una fila por defecto."""
    registros = []
    rutas_txt = sorted(glob.glob(os.path.join(ruta_datos, "group*", "*_not", "*.txt")))
    print(f"Archivos de anotación encontrados: {len(rutas_txt)}")

    for ruta_txt in rutas_txt:
        nombre_img = os.path.basename(ruta_txt).replace(".txt", "_test.jpg")
        ruta_img = ruta_txt.replace("_not", "").replace(".txt", "_test.jpg")
        grupo = os.path.basename(os.path.dirname(os.path.dirname(ruta_txt)))

        with open(ruta_txt) as f:
            for linea in f:
                partes = linea.split()
                if len(partes) != 5:          # línea malformada → se descarta aquí
                    continue
                x1, y1, x2, y2, tipo = map(int, partes)
                registros.append({
                    "imagen":   nombre_img,
                    "grupo":    grupo,
                    "tipo_id":  tipo,
                    "tipo":     MAPEO_TIPOS.get(tipo, f"desconocido({tipo})"),
                    "x1": x1, "y1": y1, "x2": x2, "y2": y2,
                    "ancho":    x2 - x1,
                    "alto":     y2 - y1,
                    "area_px":  (x2 - x1) * (y2 - y1),
                    "cx":       (x1 + x2) / 2,   # centroide del defecto
                    "cy":       (y1 + y2) / 2,
                    "ruta_img": ruta_img,
                })
    return pd.DataFrame(registros)

df_crudo = cargar_anotaciones(RUTA_DATOS)
print(f"\nFilas cargadas: {len(df_crudo):,}")
df_crudo.head(10)

In [ ]:
# Vistazo general a la estructura del DataFrame crudo
df_crudo.info()

## 3. Limpieza y validación de datos

Aplicamos una lista de verificación completa y **documentamos qué se encontró y qué se corrigió**:
nulos, duplicados, coordenadas inválidas, tipos de dato y rangos fuera de especificación.

In [ ]:
# ============================================================
# CHECK 1 — Valores nulos
# ============================================================
nulos = df_crudo.isnull().sum()
print("Valores nulos por columna:")
print(nulos[nulos > 0] if nulos.any() else "  (ninguno)")

In [ ]:
# ============================================================
# CHECK 2 — Registros duplicados
# (mismo defecto anotado dos veces en la misma imagen)
# ============================================================
llave = ["imagen", "tipo_id", "x1", "y1", "x2", "y2"]
n_dup = df_crudo.duplicated(subset=llave).sum()
print(f"Registros duplicados: {n_dup}")

In [ ]:
# ============================================================
# CHECK 3 — Coordenadas inválidas (bounding boxes degenerados)
# Un bbox es inválido si x2 <= x1 o y2 <= y1 (ancho o alto <= 0)
# ============================================================
invalidos = (df_crudo.x2 <= df_crudo.x1) | (df_crudo.y2 <= df_crudo.y1)
print(f"Bounding boxes degenerados: {invalidos.sum()}")
if invalidos.any():
    print(df_crudo.loc[invalidos, ["imagen", "x1", "y1", "x2", "y2"]].head())

In [ ]:
# ============================================================
# CHECK 4 — Tipos de defecto fuera del rango válido (1–6)
# ============================================================
tipos_malos = ~df_crudo.tipo_id.between(1, 6)
print(f"Registros con tipo fuera de rango: {tipos_malos.sum()}")
print("\nTipos de defecto presentes en los datos crudos:")
print(df_crudo.tipo.value_counts())

In [ ]:
# ============================================================
# CHECK 5 — Coherencia espacial: ¿los defectos caen dentro de la imagen 640x640?
# ============================================================
fuera = (df_crudo[["x1", "y1"]] < 0).any(axis=1) | (df_crudo[["x2", "y2"]] > 640).any(axis=1)
print(f"Defectos fuera de los límites de la imagen: {fuera.sum()}")

In [ ]:
# ============================================================
# APLICAR LA LIMPIEZA (en orden de severidad)
# ============================================================
df = df_crudo.copy()
reporte = {"Filas crudas": len(df)}

# 1. Duplicados exactos → conservar la primera aparición
df = df.drop_duplicates(subset=["imagen", "tipo_id", "x1", "y1", "x2", "y2"], keep="first")
reporte["Duplicados eliminados"] = reporte["Filas crudas"] - len(df)

# 2. Bounding boxes degenerados → eliminar
df = df[(df.x2 > df.x1) & (df.y2 > df.y1)]
reporte["BBoxes degenerados eliminados"] = reporte["Filas crudas"] - reporte["Duplicados eliminados"] - len(df)

# 3. Tipos inválidos → eliminar (si existieran)
df = df[df.tipo_id.between(1, 6)]

df = df.reset_index(drop=True)
reporte["Filas finales"] = len(df)

from IPython.display import display
print("Resumen de la limpieza:")
display(pd.Series(reporte).to_frame("filas"))

# Comparación antes/después
print("\nConteo por tipo — ANTES de limpiar:")
print(df_crudo.tipo.value_counts())
print("\nConteo por tipo — DESPUÉS de limpiar:")
print(df.tipo.value_counts())

In [ ]:
# Verificación final: el DataFrame limpio debe pasar todos los checks
assert df.isnull().sum().sum() == 0, "quedan nulos"
assert df.duplicated(subset=["imagen", "tipo_id", "x1", "y1", "x2", "y2"]).sum() == 0, "quedan duplicados"
assert ((df.x2 <= df.x1) | (df.y2 <= df.y1)).sum() == 0, "quedan bboxes degenerados"
assert df.tipo_id.between(1, 6).all(), "quedan tipos inválidos"
print("✓ Todos los checks de calidad pasaron. DataFrame listo para el análisis.")
print(f"\nShape final: {df.shape}")

## 4. Análisis univariado (una variable a la vez)

Estudiamos la distribución de cada variable por separado: tipos de defecto,
tamaños y posiciones.

In [ ]:

df.describe()[["x1", "y1", "x2", "y2", "ancho", "alto", "area_px", "cx", "cy"]]

In [ ]:
# Distribución de la variable categórica: frecuencia absoluta y relativa
freq = df.tipo.value_counts().to_frame("n_defectos")
freq["% del total"] = (freq.n_defectos / len(df) * 100).round(2)
print("Frecuencia por tipo de defecto:")
freq

In [ ]:
# Asimetría de las variables numéricas (¿son simétricas o sesgadas?)
asimetria = df[["ancho", "alto", "area_px"]].agg(["median", "mean", "std"]).T
asimetria["coef_variacion"] = (asimetria["std"] / asimetria["mean"])
print("Media vs. mediana (media >> mediana ⇒ cola derecha):")
asimetria

In [ ]:
# Defectos por imagen: cuántos defectos contiene cada placa inspeccionada
defectos_por_img = df.groupby("imagen").size()
print("Distribución de defectos por imagen:")
print(defectos_por_img.describe())

## 5. Análisis bivariado (relaciones entre variables)

Respondemos: *¿el tamaño de los defectos depende de su tipo? ¿dónde ocurre cada tipo?*

In [ ]:

# Tabla resumen por tipo de defecto
resumen = df.groupby("tipo").agg(
    n_defectos=("area_px", "count"),
    ancho_med=("ancho", "median"),
    alto_med=("alto", "median"),
    area_mediana=("area_px", "median"),
    area_p25=("area_px", lambda s: s.quantile(0.25)),
    area_p75=("area_px", lambda s: s.quantile(0.75)),
).sort_values("n_defectos", ascending=False)
resumen

In [ ]:
# Relación entre tipo y posición: centroide promedio de cada tipo de defecto
posicion = df.groupby("tipo").agg(cx_prom=("cx", "mean"), cy_prom=("cy", "mean"),
                                  cx_std=("cx", "std"), cy_std=("cy", "std")).round(1)
print("Posición promedio del defecto por tipo (la imagen es 640x640):")
posicion

In [ ]:
# Correlaciones entre variables numéricas
matriz = df[["ancho", "alto", "area_px", "cx", "cy"]].corr().round(3)
print("Matriz de correlación (Pearson):")
matriz

## 6. Visualizaciones

### Gráfico 1 — Distribución de defectos por tipo (análisis univariado)

Responde: *¿qué defectos son más frecuentes y las clases están balanceadas
para entrenar un modelo de IA sin sesgo?*

In [ ]:

orden = df.tipo.value_counts().index
ax = df.tipo.value_counts().reindex(orden).plot.barh(color="#2a7fb8", edgecolor="black")
ax.set_title("Distribución de defectos por tipo (dataset DeepPCB)")
ax.set_xlabel("Número de defectos anotados")
ax.set_ylabel("")
for i, v in enumerate(df.tipo.value_counts().reindex(orden)):
    ax.text(v + 30, i, f"{v:,}", va="center")
plt.tight_layout()
plt.show()

### Gráfico 2 — Tamaño de los defectos por tipo (análisis bivariado)

Responde: *¿qué tan pequeños son los defectos? Los objetos pequeños son el mayor
reto técnico para los modelos de detección con IA.*

In [ ]:

tipos = resumen.index.tolist()
datos = [df.loc[df.tipo == t, "area_px"] for t in tipos]
fig, ax = plt.subplots(figsize=(11, 5))
bp = ax.boxplot(datos, labels=tipos, showfliers=False, patch_artist=True)
for parche in bp["boxes"]:
    parche.set_facecolor("#2a7fb8")
ax.set_yscale("log")
ax.set_title("Área del bounding box por tipo de defecto (escala logarítmica, sin atípicos)")
ax.set_xlabel("Tipo de defecto")
ax.set_ylabel("Área (px², escala log)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

### Gráfico 3 — Defectos por imagen (análisis univariado)

Responde: *¿cuántas decisiones debe tomar el modelo de IA por placa inspeccionada?*

In [ ]:

ax = defectos_por_img.plot.hist(bins=25, color="#2a7fb8", edgecolor="black")
ax.set_title("Número de defectos por imagen de placa (640×640 px)")
ax.set_xlabel("Defectos por imagen")
ax.set_ylabel("Número de imágenes")
ax.axvline(defectos_por_img.mean(), color="crimson", linestyle="--",
           label=f"Media = {defectos_por_img.mean():.1f}")
ax.axvline(defectos_por_img.median(), color="darkorange", linestyle=":",
           label=f"Mediana = {defectos_por_img.median():.0f}")
ax.legend()
plt.tight_layout()
plt.show()

### Gráfico 4 — Mapa de calor: ¿dónde se concentran los defectos? (análisis espacial)

Responde: *¿hay zonas de la placa más propensas a fallas? Esto indica qué contexto
espacial debe aprender el modelo de IA para localizar defectos.*

In [ ]:

# Usamos una placa real como fondo para dar referencia espacial
import matplotlib.image as mpimg
img_fondo = mpimg.imread(df.ruta_img.iloc[0])

fig, ax = plt.subplots(figsize=(8, 8))
hb = ax.hexbin(df.cx, df.cy, gridsize=25, cmap="hot_r", mincnt=1)
fig.colorbar(hb, ax=ax, label="Número de defectos")
ax.imshow(img_fondo, alpha=0.35, cmap="gray")
ax.set_title("Concentración espacial de los 10,013 defectos sobre una placa de ejemplo")
ax.set_xlabel("x (px)")
ax.set_ylabel("y (px)")
ax.grid(False)
plt.tight_layout()
plt.show()

### BONUS — Gráfico 5: dispersión posición × tipo (análisis bivariado espacial)

Cada color es un tipo de defecto sobre la placa: *¿cada tipo aparece en zonas
distintas o todos se mezclan?*

In [ ]:

fig, ax = plt.subplots(figsize=(8, 8))
colores = {"open": "red", "short": "orange", "mousebite": "limegreen",
           "spur": "deepskyblue", "spurious copper": "magenta", "pin-hole": "gold"}
for tipo, color in colores.items():
    sub = df[df.tipo == tipo]
    ax.scatter(sub.cx, sub.cy, s=6, alpha=0.25, color=color, label=f"{tipo} (n={len(sub):,})")
ax.set_title("Ubicación de cada defecto según su tipo")
ax.set_xlabel("x (px)"); ax.set_ylabel("y (px)")
ax.set_xlim(0, 640); ax.set_ylim(640, 0)   # orientación imagen
ax.legend(markerscale=3, fontsize=9)
plt.tight_layout()
plt.show()

### BONUS — Ejemplo visual: placa con defectos vs. plantilla sin defectos

Así se ve el problema que resuelve la IA: comparar la placa contra su plantilla
y localizar las diferencias (los defectos anotados).

In [ ]:

ejemplo = df.imagen.iloc[0]
ruta_test = df.ruta_img.iloc[0]
ruta_temp = ruta_test.replace("_test", "_temp")
recortes = df[df.imagen == ejemplo]

fig, ejes = plt.subplots(1, 2, figsize=(14, 7))
for ax, ruta, titulo in [(ejes[0], ruta_test, f"Placa con defectos: {ejemplo}"),
                          (ejes[1], ruta_temp, "Plantilla (sin defectos)")]:
    ax.imshow(mpimg.imread(ruta), cmap="gray")
    ax.set_title(titulo)
    ax.axis("off")

for _, r in recortes.iterrows():
    ejes[0].add_patch(plt.Rectangle((r.x1, r.y1), r.ancho, r.alto,
                                    fill=False, edgecolor=colores[r.tipo], lw=1.6))
    ejes[0].text(r.x1, r.y1 - 4, r.tipo, color=colores[r.tipo], fontsize=7)
plt.tight_layout()
plt.show()

## 7. Conclusiones

1. **La inspección de PCB con IA es viable.** El dataset DeepPCB demuestra el paradigma de comparación placa-vs-plantilla; el paper original reporta **98.6% mAP y 98.2% F-score** con una red neuronal profunda sobre estos mismos datos, y el sistema corre a 62 FPS (suficiente para línea de producción).
2. **Calidad de los datos:** de 10,013 registros crudos, el dataset resultó limpio (0 nulos, 0 duplicados, 0 bounding boxes degenerados); la limpieza validó la integridad de las anotaciones.
3. **Clases casi balanceadas** (mousebite 1,965 vs. spurious copper 1,474, diferencia < 25%): un modelo de IA entrenado aquí no tendrá sesgo fuerte hacia una clase mayoritaria.
4. **Defectos pequeños:** la mediana de área por tipo está entre cientos y miles de px² sobre imágenes de 640×640 (objetos diminutos, con alto coeficiente de variación y colas derechas). Esto justifica arquitecturas especializadas en objetos pequeños (TDD-net, YOLO con alta resolución).
5. **Múltiples decisiones por placa:** en promedio ~6.7 defectos por imagen (rango 1–15). El sistema de IA no da un solo veredicto: debe localizar y clasificar todos los defectos y recién entonces decidir si la placa **pasa o se rechaza**.
6. **Distribución espacial:** los defectos se mezclan por tipo en todo el tablero (no hay zonas exclusivas por tipo), por lo que el modelo necesita contexto espacial global, no reglas por región.

## Limitaciones

- **Defectos sintéticos:** fueron agregados digitalmente (Photoshop) sobre imágenes reales; un modelo entrenado aquí puede no generalizar a defectos reales de fábrica (iluminación, suciedad, variaciones de proceso).
- **Alcance:** el dataset cubre defectos de *fabricación del circuito impreso* (trazas, cobre, agujeros), **no** de *ensamblaje SMT* (componentes faltantes, soldaduras defectuosas). Para la propuesta completa se complementaría con datasets como **PCB-AoI** (inspección de pasta de soldadura).
- **Un solo diseño de placa:** todas las imágenes corresponden al mismo PCB; la generalización a otros diseños no está garantizada.
- **Uso académico:** la licencia del dataset original restringe su uso a investigación.

## Referencias

- Ding, R., Dai, L., Li, G. & Liu, H. (2019). *TDD-net: a tiny defect detection network for printed circuit boards*. CAAI Transactions on Intelligence Technology, 4(2), 110–116. https://doi.org/10.1049/trit.2019.0019
- Huang, W. & Wei, P. (2019). *A PCB Dataset for Defects Detection and Classification*. arXiv:1901.08204.
- Repositorio del dataset: https://github.com/tangsanli5201/DeepPCB